# Notas — Aula 10: Polimorfismo e delegação

Marco: nasce `mover()` no `Robo` base. Primeiro ele já responde diferente em
`RoboVeloz`/`RoboExplorador` **sem precisar ser sobrescrito** — só chama
`self.avancar()`, e o despacho dinâmico resolve para a versão certa (polimorfismo por
herança). Um objeto **fora** da hierarquia de `Robo` entra na mesma lista e responde a
`mover()` do seu próprio jeito (duck typing). Depois, `mover()` é refatorado para
**delegar** a decisão de como andar a um objeto composto, `self.estrategia` —
trocável em tempo de execução, sem subclasse nenhuma.

> ⚠️ **Antes de começar:** rode todas as células a partir do topo (**Run All**) — cada
> seção depende da anterior.

In [1]:
LADO_GRADE = 10

## Duck typing — mesmo método, sem herança nenhuma

Em Java ou C#, tratar dois tipos de forma uniforme normalmente exige uma interface
comum. Em Python não: se um objeto tem o método com o nome certo, ele serve — não
importa a árvore de classes. Chama-se **duck typing**: "se anda como pato e grasna
como pato, eu trato como pato". O contrato é **implícito** — não existe `interface`
nem `abstract` para declarar. No robô, é exatamente essa flexibilidade que vai deixar
um objeto **sem nenhum parentesco** com `Robo` entrar na mesma lista de `mover()`.

In [2]:
class Gato:
    def falar(self):
        return "Miau"


class Cachorro:
    def falar(self):
        return "Au au"


animais = [Gato(), Cachorro()]
for a in animais:
    print(a.falar())

Miau
Au au


In [3]:
class NotificadorEmail:
    def enviar(self, msg):
        print(f"[EMAIL] {msg}")


class NotificadorSMS:
    def enviar(self, msg):
        print(f"[SMS] {msg}")


class NotificadorPush:
    def enviar(self, msg):
        print(f"[PUSH] {msg}")


def broadcast(lista, msg):
    for n in lista:
        n.enviar(msg)


broadcast([NotificadorEmail(), NotificadorSMS(), NotificadorPush()],
          "Sistema fora do ar às 22h")

[EMAIL] Sistema fora do ar às 22h
[SMS] Sistema fora do ar às 22h
[PUSH] Sistema fora do ar às 22h


### Sua vez

Complete `NotificadorSlack`: mesmo contrato dos outros (`enviar(self, msg)`), mas
imprime `f"[SLACK] {msg}"`. Depois, `broadcast` já é chamado com uma lista de quatro
notificadores (os três de cima + o novo) — sem mudar uma linha de `broadcast`.

*Dica: copie a forma de `NotificadorPush`, só troca o texto entre colchetes.*

In [4]:
class NotificadorSlack:
    def enviar(self, msg):
        # TODO: imprima f"[SLACK] {msg}"
        ...


broadcast([NotificadorEmail(), NotificadorSMS(), NotificadorPush(), NotificadorSlack()],
          "Deploy concluído")

[EMAIL] Deploy concluído
[SMS] Deploy concluído
[PUSH] Deploy concluído


## `mover()` no robô — polimorfismo por herança

`RoboVeloz` sempre avança 2 casas, um `Robo` comum avança 1, `RoboExplorador` avança 1
mas também anota onde já passou — três comportamentos diferentes atrás do mesmo nome
de método, `avancar()`. Agora `Robo` ganha `mover()`, que só faz
`return self.avancar()`. Ele nunca é sobrescrito em nenhuma subclasse — mesmo assim o
resultado difere, porque dentro dele `self.avancar()` usa o `self` de quem chamou.
Isso é **despacho dinâmico**: o Python decide, em tempo de execução, qual `avancar()`
rodar, olhando o tipo real do objeto — não o que está escrito no código de `mover()`.

In [5]:
from enum import Enum


class Direcao(Enum):
    LESTE = (1, 0)
    NORTE = (0, 1)
    OESTE = (-1, 0)
    SUL = (0, -1)


class Robo:
    LADO_GRADE = 10

    def __init__(self, nome, x=0, y=0, direcao=Direcao.LESTE, obstaculos=None):
        self.nome = nome
        self.x = x
        self.y = y
        self.direcao = direcao
        self.obstaculos = obstaculos if obstaculos is not None else {}

    def sensor_frente(self):
        dx, dy = self.direcao.value
        nx, ny = self.x + dx, self.y + dy
        return 0 <= nx < Robo.LADO_GRADE and 0 <= ny < Robo.LADO_GRADE and (nx, ny) not in self.obstaculos

    def avancar(self):
        if self.sensor_frente():
            dx, dy = self.direcao.value
            self.x += dx
            self.y += dy
            return True
        return False

    def mover(self):
        return self.avancar()


class RoboVeloz(Robo):
    def avancar(self):
        moveu1 = super().avancar()
        moveu2 = super().avancar()
        return moveu1 or moveu2


class RoboExplorador(Robo):
    def __init__(self, nome, **kwargs):
        super().__init__(nome, **kwargs)
        self.areas_visitadas = {(self.x, self.y)}

    def avancar(self):
        moveu = super().avancar()
        if moveu:
            self.areas_visitadas.add((self.x, self.y))
        return moveu


robos = [Robo("Wall-E"), RoboVeloz("Flash"), RoboExplorador("Curioso")]
for r in robos:
    r.mover()
    print(r.nome, r.x, r.y)

Wall-E 1 0
Flash 2 0
Curioso 1 0


### Sua vez

Complete `robo_mais_rapido(lista)`: chame `.mover()` em cada objeto da lista e
devolva o **objeto** (não o nome) que ficou com o maior `x` depois de mover.

*Dica: `max(lista, key=lambda r: r.x)` — mas só depois de mover todo mundo.*

In [6]:
def robo_mais_rapido(lista):
    # TODO: chame .mover() em cada item de lista; devolva o objeto com maior x
    ...


robos2 = [Robo("Wall-E"), RoboVeloz("Flash"), RoboExplorador("Curioso")]
vencedor = robo_mais_rapido(robos2)
print(vencedor.nome if vencedor else None)

None


## `isinstance` vs. duck typing

E se, em vez de confiar no despacho automático, alguém decidisse "na mão" o que cada
tipo faz, com uma cadeia de `isinstance`? O resultado é o mesmo — só que escrito em
mais código, e com um problema esperando: um objeto **fora** da árvore de `Robo`, como
`RoboSimulado`, tem exatamente o método que precisa (`mover()`) mas é rejeitado, porque
`isinstance` pergunta "de que família você é", não "você sabe fazer isso". `isinstance`
não é ruim em si — ele é legítimo dentro de um método como `__eq__`, checando se a
**comparação** faz sentido (Aula 7, `Posicao.__eq__`); o problema é usá-lo **por fora**,
competindo com o polimorfismo que já resolve sozinho.

In [7]:
def mover_com_isinstance(objeto):
    if isinstance(objeto, RoboVeloz):
        objeto.mover()
    elif isinstance(objeto, RoboExplorador):
        objeto.mover()
    elif isinstance(objeto, Robo):
        objeto.mover()
    else:
        print(f"Não sei mover {objeto.nome}")


class RoboSimulado:
    def __init__(self, nome):
        self.nome = nome
        self.passos = 0

    def mover(self):
        self.passos += 1
        print(f"{self.nome} (simulado) deu um passo virtual. Total: {self.passos}")
        return True


robos3 = [Robo("Wall-E"), RoboVeloz("Flash"), RoboExplorador("Curioso"), RoboSimulado("Sim-1")]

print("--- loop polimórfico (sem isinstance) ---")
for r in robos3:
    r.mover()

print("--- loop com isinstance ---")
for r in robos3:
    mover_com_isinstance(r)

--- loop polimórfico (sem isinstance) ---
Sim-1 (simulado) deu um passo virtual. Total: 1
--- loop com isinstance ---
Não sei mover Sim-1


### Sua vez

Complete `RoboEco`, um duck-typed **fora** da hierarquia de `Robo` (não herda de
nada): `mover(self)` deve imprimir `f"{self.nome} ecoa um bip"` e devolver `True`. Ele
não precisa saber nada sobre grade, obstáculo ou direção — só ter o método com a cara
certa. Ele já é acrescentado à lista `robos3` abaixo — confirme que o loop polimórfico
continua funcionando sem tocar em `mover_com_isinstance`.

*Dica: mesma estrutura de `RoboSimulado` acima, sem o contador de passos.*

In [8]:
class RoboEco:
    def __init__(self, nome):
        self.nome = nome

    def mover(self):
        # TODO: imprima f"{self.nome} ecoa um bip" e devolva True
        ...


robos3.append(RoboEco("Echo-1"))
for r in robos3:
    r.mover()

Sim-1 (simulado) deu um passo virtual. Total: 2


## O preço do duck typing: erro só em tempo de execução

Duck typing troca a segurança de tipo em tempo de compilação por flexibilidade — e
isso tem um preço: o erro só aparece quando o método realmente falta, não antes.
Numa linguagem estaticamente tipada, o compilador provavelmente pegaria isso mais
cedo; aqui, o `for` só descobre o problema quando chega no objeto errado.

In [9]:
robos3.append(("Caixa", 0, 0))

try:
    for r in robos3:
        r.mover()
except AttributeError as erro:
    print(f"AttributeError: {erro}")

Sim-1 (simulado) deu um passo virtual. Total: 3
AttributeError: 'tuple' object has no attribute 'mover'


## Delegação — mecânica básica, fora do robô

Composição já é conhecida: um atributo que é, ele mesmo, um objeto. **Delegação** é o
nome novo: quando um método não faz o trabalho sozinho, só repassa a chamada para o
objeto que ele carrega. `ProcessadorImagens` não sabe nada sobre PNG ou JPG — ele só
confia que o `backend` recebido no construtor tem um `.salvar(caminho)`. Trocar o
backend, de fora, muda o comportamento inteiro sem mudar uma linha de
`ProcessadorImagens`.

In [10]:
class BackendPNG:
    def salvar(self, caminho):
        print(f"[PNG] Salvando em {caminho}")


class BackendJPG:
    def salvar(self, caminho):
        print(f"[JPG] Salvando em {caminho}")


class ProcessadorImagens:
    def __init__(self, backend):
        self.backend = backend           # composição: ProcessadorImagens TEM um backend

    def salvar_imagem(self, caminho):
        self.backend.salvar(caminho)     # delegação: repassa a chamada pro backend


proc_png = ProcessadorImagens(BackendPNG())
proc_jpg = ProcessadorImagens(BackendJPG())
proc_png.salvar_imagem("foto.png")
proc_jpg.salvar_imagem("foto.jpg")

[PNG] Salvando em foto.png
[JPG] Salvando em foto.jpg


### Sua vez

Complete `BackendGIF`, mesmo contrato dos outros backends (`salvar(self, caminho)`),
imprimindo `f"[GIF] Salvando em {caminho}"`. Um `ProcessadorImagens` com esse backend
já é criado e chamado abaixo — sem mudar uma linha de `ProcessadorImagens`.

*Dica: copie a forma de `BackendJPG`, só troca o rótulo e a extensão.*

In [11]:
class BackendGIF:
    def salvar(self, caminho):
        # TODO: imprima f"[GIF] Salvando em {caminho}"
        ...


proc_gif = ProcessadorImagens(BackendGIF())
proc_gif.salvar_imagem("foto.gif")

## `Robo` passa a delegar `mover()` para uma `estrategia`

`mover()` hoje faz `return self.avancar()` direto. Agora essa linha sai do `Robo` e
vai para uma classe separada, `EstrategiaPadrao`, com `mover(self, robo)` — note que é
a **estratégia** que recebe o robô como argumento, não o contrário. `Robo.__init__`
ganha uma `estrategia`, com `EstrategiaPadrao()` como padrão (mesmo truque de
`obstaculos=None` que já usamos desde a Aula 1), e `mover()` deixa de calcular e passa
a **delegar**: `return self.estrategia.mover(self)`.

In [12]:
ORDEM_DIRECOES = [Direcao.LESTE, Direcao.NORTE, Direcao.OESTE, Direcao.SUL]


class EstrategiaPadrao:
    def mover(self, robo):
        return robo.avancar()


class Robo:
    LADO_GRADE = 10

    def __init__(self, nome, x=0, y=0, direcao=Direcao.LESTE, obstaculos=None, estrategia=None):
        self.nome = nome
        self.x = x
        self.y = y
        self.direcao = direcao
        self.obstaculos = obstaculos if obstaculos is not None else {}
        self.estrategia = estrategia if estrategia is not None else EstrategiaPadrao()

    def sensor_frente(self):
        dx, dy = self.direcao.value
        nx, ny = self.x + dx, self.y + dy
        return 0 <= nx < Robo.LADO_GRADE and 0 <= ny < Robo.LADO_GRADE and (nx, ny) not in self.obstaculos

    def avancar(self):
        if self.sensor_frente():
            dx, dy = self.direcao.value
            self.x += dx
            self.y += dy
            return True
        return False

    def girar(self, lado):
        if lado == "ESQ":
            self.direcao = ORDEM_DIRECOES[(ORDEM_DIRECOES.index(self.direcao) + 1) % 4]
        elif lado == "DIR":
            self.direcao = ORDEM_DIRECOES[(ORDEM_DIRECOES.index(self.direcao) - 1) % 4]

    def mover(self):
        return self.estrategia.mover(self)


r = Robo("Teste")
print(type(r.estrategia).__name__)
r.mover()
print(r.x, r.y)

EstrategiaPadrao
1 0


### Erro clássico: esquecer os parênteses ao trocar a peça

Assim como `self.sensor = Sensor` (sem chamar) na Aula 9, esquecer de **instanciar**
a estratégia não dá erro na hora de criar o robô — só quando alguém tenta usá-la.

In [13]:
class RoboQuebrado(Robo):
    def __init__(self, nome, **kwargs):
        super().__init__(nome, **kwargs)
        self.estrategia = EstrategiaPadrao      # esqueceu os parênteses!


try:
    rq = RoboQuebrado("Bug")
    rq.mover()
except TypeError as erro:
    print(f"TypeError: {erro}")

TypeError: EstrategiaPadrao.mover() missing 1 required positional argument: 'robo'


### As duas camadas de polimorfismo, lado a lado

`RoboVeloz` e `RoboExplorador` continuam existindo, com `avancar()` sobrescrito.
Mesmo com `estrategia` do mesmo tipo (`EstrategiaPadrao`) nos três, eles andam
diferente — porque a diferença vem de `avancar()` (herança), não de `estrategia`
(composição, igual nos três por enquanto). `estrategia` decide **quando/como** chamar
`avancar()`; a subclasse decide **o que** `avancar()` faz de fato — as duas camadas
convivem sem conflito.

In [14]:
class RoboVeloz(Robo):
    def avancar(self):
        moveu1 = super().avancar()
        moveu2 = super().avancar()
        return moveu1 or moveu2


class RoboExplorador(Robo):
    def __init__(self, nome, **kwargs):
        super().__init__(nome, **kwargs)
        self.areas_visitadas = {(self.x, self.y)}

    def avancar(self):
        moveu = super().avancar()
        if moveu:
            self.areas_visitadas.add((self.x, self.y))
        return moveu


robos4 = [Robo("Wall-E"), RoboVeloz("Flash"), RoboExplorador("Curioso")]
for rb in robos4:
    rb.mover()
    print(rb.nome, rb.x, rb.y, "—", type(rb.estrategia).__name__)

Wall-E 1 0 — EstrategiaPadrao
Flash 2 0 — EstrategiaPadrao
Curioso 1 0 — EstrategiaPadrao


### Sua vez

Complete `trocar_estrategia(robo, nova_estrategia)`: troque `robo.estrategia` pela
`nova_estrategia` recebida e, na sequência, chame `robo.mover()` — devolvendo o que
`mover()` devolver.

*Dica: duas linhas — uma atribuição e um `return robo.mover()`.*

In [15]:
def trocar_estrategia(robo, nova_estrategia):
    # TODO: troque robo.estrategia pela nova_estrategia e retorne robo.mover()
    ...


r2 = Robo("Bender")
resultado = trocar_estrategia(r2, EstrategiaPadrao())
print(type(r2.estrategia).__name__, resultado, r2.x)

EstrategiaPadrao None 0


## `EstrategiaEsquiva` — a mesma peça, comportamento realmente diferente

`EstrategiaPadrao` só anda para a frente e desiste se bater. `EstrategiaEsquiva` gira
antes de desistir: enquanto o sensor não estiver livre (até 4 tentativas — depois de 4
giros de 90° já testou os quatro lados), gira para a direita; ao sair do laço, avança.
Ela também só tem `mover(self, robo)`, mesma assinatura de `EstrategiaPadrao`, sem
herdar dela — duck typing de novo, agora por dentro do `Robo`. Trocar
`robo.estrategia` no meio da vida do objeto muda o jeito de andar **sem criar `Robo`
novo, sem subclasse nenhuma** — igual `robo.sensor = Sensor(5)` fez na Aula 9.

In [16]:
class EstrategiaEsquiva:
    def mover(self, robo):
        tentativas = 0
        while not robo.sensor_frente() and tentativas < 4:
            robo.girar("DIR")
            tentativas += 1
        return robo.avancar()


obstaculos = {(1, 0): True}
robo1 = Robo("Wall-E", obstaculos=obstaculos)

robo1.mover()
print(robo1.x, robo1.y, "— parou?", robo1.x == 0)

robo1.estrategia = EstrategiaEsquiva()
robo1.mover()
print(robo1.x, robo1.y, "— desviou?", robo1.direcao != Direcao.LESTE)

0 0 — parou? True
0 1 — desviou? True


### Sua vez

Complete `EstrategiaParada`: `mover(self, robo)` **nunca** chama `avancar()` — só
devolve `False` sempre, não importa o estado do robô. Parece inútil, mas prova o
ponto: qualquer objeto com a assinatura certa serve de estratégia, mesmo um que não
faz nada. Um robô livre (sem obstáculo) já é testado com essa estratégia abaixo —
confirme que ele não sai do lugar.

*Dica: um método de uma linha só — `return False`.*

In [17]:
class EstrategiaParada:
    def mover(self, robo):
        # TODO: nunca avance — apenas devolva False
        ...


robo6 = Robo("Parado")
robo6.estrategia = EstrategiaParada()
resultado = robo6.mover()
print(resultado, robo6.x, robo6.y)

None 0 0


## Para aprofundar

- Duck typing (definição oficial) — glossário da documentação oficial: https://docs.python.org/3/glossary.html#term-duck-typing
- Polimorfismo em Python — W3Schools: https://www.w3schools.com/python/python_polymorphism.asp
- Classes e herança (revisão) — Tutorial oficial: https://docs.python.org/3/tutorial/classes.html
- Composição e delegação — Real Python: https://realpython.com/inheritance-composition-python/
- `functools.singledispatch` (quando o despacho por tipo é mesmo necessário) — documentação oficial: https://docs.python.org/3/library/functools.html#functools.singledispatch